# Лабораторна 6. Власна RAG-система: документація Tailwind CSS

Система відповідає українською на питання про Tailwind CSS v4 за його англомовною документацією і вказує сторінки, з яких узято відповідь.
Розділи названо так само, як пункти завдання (1.1–1.8).

In [3]:
!pip install -q "sentence-transformers>=5" langchain-text-splitters rank_bm25 faiss-cpu

In [4]:
import re
import subprocess
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Пристрій: {device}")

Пристрій: cuda


## 1.1. Корпус

**Джерело.** Офіційна документація Tailwind CSS v4: репозиторій tailwindlabs/tailwindcss.com, папка `src/docs`, зафіксований коміт `7f92c22`.

**Ліцензія.** Репозиторій сайту не має відкритої ліцензії, його код доступний лише для ознайомлення. Тому корпус я не кладу у свій репозиторій: ноутбук сам завантажує файли за зафіксованим комітом.

**Склад.** 39 сторінок у форматі MDX: 13 оглядових (стилізація класами, стани, адаптивність, темна тема, тема, директиви, Preflight, оновлення з v3 на v4, сумісність, налаштування редактора та інші) і 26 сторінок окремих утиліт (flex, grid, padding, font-size, animation та ін.). Після очищення від JSX-блоків залишається чистий текст на 251 КБ.

**Хто питає і чому RAG.** Питає розробник, який верстає на Tailwind і шукає конкретну деталь: яка директива, як налаштувати, що змінилося у v4. Питання він ставить українською, а документація англійською. Цілком у промпт її не покласти: сторінок багато, а моделі потрібні лише кілька абзаців. Fine-tuning теж не підходить: документація змінюється з кожним релізом, а модель мусить показувати, з якої сторінки взято відповідь. RAG дає це без перенавчання: достатньо оновити корпус.

In [5]:
COMMIT = "7f92c2213315c195dae583d68752da4042da3ade"
REPO_URL = "https://github.com/tailwindlabs/tailwindcss.com.git"
DOC_NAMES = [
    "styling-with-utility-classes", "hover-focus-and-other-states", "responsive-design", "dark-mode", "theme", "colors",
    "adding-custom-styles", "detecting-classes-in-source-files", "functions-and-directives", "preflight",
    "upgrade-guide", "compatibility", "editor-setup", "flex", "flex-direction", "flex-wrap", "grid-template-columns",
    "gap", "justify-content", "align-items", "padding", "margin", "width", "font-size", "font-weight", "border-radius",
    "box-shadow", "opacity", "display", "position", "z-index", "aspect-ratio", "line-clamp", "overflow", "cursor",
    "transition-property", "transition-duration", "animation", "text-wrap",
]

REPO_DIR = Path("tailwindcss.com")
if not REPO_DIR.exists():
    # Беремо конкретний коміт, щоб корпус і бенчмарк не змінювалися
    run = lambda cmd: subprocess.run(cmd, shell=True, check=True)
    run(f"mkdir -p {REPO_DIR} && cd {REPO_DIR} && git init -q && git remote add origin {REPO_URL}")
    run(f"cd {REPO_DIR} && git fetch -q --depth 1 origin {COMMIT} && git checkout -q FETCH_HEAD")
DOCS_DIR = REPO_DIR / "src" / "docs"


def clean(raw):
    # Документація написана в MDX: прибираємо імпорти, демонстраційні блоки JSX і службові коментарі
    text = re.sub(r"^(import|export) .*?;\s*$", "", raw, flags=re.M)

    def api_table(match):
        rows = re.findall(r'\[\s*"((?:[^"\\]|\\.)*)"\s*,\s*"((?:[^"\\]|\\.)*)"\s*(?:,\s*"[^"]*"\s*)?\]', match.group(0))
        return "\n".join(f"- `{utility}` → `{css}`" for utility, css in rows) + "\n"

    text = re.sub(r"<ApiTable.*?/>", api_table, text, flags=re.S)
    text = re.sub(r"<Example\b.*?</Example>", "", text, flags=re.S)
    text = re.sub(r"<Iframe\b.*?(/>|</Iframe>)", "", text, flags=re.S)
    parts = re.split(r"(```.*?```)", text, flags=re.S)
    for i, part in enumerate(parts):
        if part.startswith("```"):
            parts[i] = re.sub(r"^```(\w+) \{\{.*?\}\}", r"```\1", part)
            continue
        part = re.sub(r"<h([234])[^>]*>\s*<a[^>]*>(.*?)</a>\s*</h\1>", lambda m: "#" * int(m.group(1)) + " " + m.group(2), part, flags=re.S)
        part = re.sub(r"\{<>(.*?)</>\}", r"\1", part, flags=re.S)
        part = re.sub(r"\{/\*.*?\*/\}", "", part, flags=re.S)
        part = re.sub(r"</?[A-Z][A-Za-z]*\b[^>]*?/?>", "", part, flags=re.S)
        part = re.sub(r"<(details|table|svg|div|ul|ol|button|figure|img)\b.*?(</\1>|/>)", "", part, flags=re.S)
        part = re.sub(r"^[{}]\s*$", "", part, flags=re.M)
        parts[i] = part
    text = "".join(parts)
    text = re.sub(r"[ \t]*<!--.*?-->[ \t]*\n?", "", text, flags=re.S)
    text = re.sub(r"[ \t]*(/\*|//|#) ?\[!code[^\]]*\]( ?\*/)?[ \t]*\n?", "", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


documents = {}
for name in DOC_NAMES:
    raw = (DOCS_DIR / f"{name}.mdx").read_text()
    title = re.search(r'export const title = "(.*?)";', raw).group(1)
    documents[name] = {"title": title, "url": f"https://tailwindcss.com/docs/{name}", "text": clean(raw)}

total_chars = sum(len(doc["text"]) for doc in documents.values())
print(f"Документів: {len(documents)}, разом {total_chars} символів ({total_chars / 1024:.0f} КБ)")
for name, doc in list(documents.items())[:6]:
    print(f"  {name}: «{doc['title']}», {len(doc['text'])} символів")

Документів: 39, разом 257471 символів (251 КБ)
  styling-with-utility-classes: «Styling with utility classes», 22388 символів
  hover-focus-and-other-states: «Hover, focus, and other states», 41677 символів
  responsive-design: «Responsive design», 10214 символів
  dark-mode: «Dark mode», 3579 символів
  theme: «Theme variables», 33234 символів
  colors: «Colors», 20491 символів


## 1.2. Бенчмарк

Бенчмарк я склав до налаштування системи за текстом документації. Це 29 питань українською: 25 мають відповідь у документах, 4 — без відповіді (ціна Tailwind Plus, Angular, автор Tailwind, версія React). До кожного питання записано тип, документ або документи з відповіддю, вираз-доказ (регулярний вираз, який має бути в правильному фрагменті) і еталонна відповідь. У більшості питань про міграцію з v3 у документах є і стара, і нова поведінка, тому вони перевіряють, чи система не плутає версії.

In [6]:
# (питання, тип, документи з відповіддю, вираз-доказ, еталонна відповідь)
BENCHMARK = [
    ("Які мінімальні версії браузерів потрібні для Tailwind CSS v4.0?", "факт",
     ["compatibility", "upgrade-guide"], r"Chrome 111", "Chrome 111, Safari 16.4 і Firefox 128."),
    ("Якою командою можна автоматично оновити проєкт із Tailwind v3 до v4?", "як зробити",
     ["upgrade-guide"], r"npx @tailwindcss/upgrade", "Командою npx @tailwindcss/upgrade."),
    ("Чим у v4 замінили директиви @tailwind base, @tailwind components і @tailwind utilities?", "міграція",
     ["upgrade-guide"], r"regular CSS `@import` statement", "Звичайним CSS-імпортом: @import \"tailwindcss\";"),
    ("Де в назві класу в v4 потрібно ставити знак оклику для модифікатора important?", "міграція",
     ["upgrade-guide"], r"very end of the class name", "У самому кінці назви класу, наприклад flex!."),
    ("Який колір межі за замовчуванням мають утиліти border-* у v4?", "міграція",
     ["upgrade-guide"], r"`currentColor` in v4", "currentColor (у v3 це був gray-200)."),
    ("Як у v4 налаштувати утиліту container, наприклад центрувати її і додати відступи?", "міграція",
     ["upgrade-guide"], r"@utility container", "Розширити її директивою @utility container з margin-inline: auto і padding-inline."),
    ("Як виглядає префікс до класів у v4, якщо задати його tw?", "міграція",
     ["upgrade-guide"], r"tw:flex", "Префікс стоїть на початку і виглядає як варіант: tw:flex, tw:bg-red-500."),
    ("Якої товщини кільце додавала утиліта ring у v3 і якою вона стала у v4?", "міграція",
     ["upgrade-guide"], r"`3px` ring", "У v3 кільце було 3px, у v4 стало 1px; щоб отримати 3px, пишуть ring-3."),
    ("Як у v4 підключити старий файл tailwind.config.js?", "міграція",
     ["upgrade-guide"], r"@config", "Явно, директивою @config із шляхом до файлу; автоматично його більше не знаходять."),
    ("Для чого призначена директива @theme?", "поняття",
     ["functions-and-directives", "theme"], r"custom design tokens", "Для оголошення власних токенів дизайну: шрифтів, кольорів, брейкпоінтів."),
    ("Яка директива додає власні утиліти, що працюють з варіантами hover, focus і lg?", "клас або директива",
     ["functions-and-directives", "adding-custom-styles"], r"add custom utilities to your project", "Директива @utility."),
    ("Для чого потрібна директива @reference у компонентах Vue чи Svelte?", "поняття",
     ["functions-and-directives"], r"import your main stylesheet for reference",
     "Вона імпортує головний файл стилів лише для довідки, щоб у \u003cstyle> працювали @apply і @variant з темою, утилітами й варіантами, а CSS не дублювався."),
    ("Чим змінні в @theme відрізняються від звичайних змінних у :root?", "поняття",
     ["theme"], r"create new utility classes", "Змінні в @theme ще й створюють утиліти і мають оголошуватися на верхньому рівні, а :root потрібен для змінних без утиліт."),
    ("Як увімкнути темну тему за допомогою класу dark замість налаштування системи?", "як зробити",
     ["dark-mode"], r"@custom-variant dark \(&:where\(\.dark", "Перевизначити варіант dark: @custom-variant dark (&:where(.dark, .dark *)); і додати клас dark до html."),
    ("Що означає mobile-first у системі брейкпоінтів Tailwind?", "поняття",
     ["responsive-design"], r"take effect on all screen sizes", "Утиліти без префікса діють на всіх екранах, а з префіксом, як md:, — на цьому брейкпоінті й більших."),
    ("Як додати власний брейкпоінт до теми?", "як зробити",
     ["responsive-design", "theme"], r"--breakpoint-", "Оголосити змінну --breakpoint-* у @theme, наприклад --breakpoint-3xl: 120rem."),
    ("Як використати container queries, щоб стилі залежали від розміру батьківського елемента?", "як зробити",
     ["responsive-design"], r"`@container` class", "Позначити батька класом @container і використовувати варіанти @sm, @md для дочірніх елементів."),
    ("Чому Tailwind не згенерує клас, якщо його ім'я складати з частин, як text-{{ error ? 'red' : 'green' }}-600?", "поняття",
     ["detecting-classes-in-source-files"], r"do not exist", "Бо Tailwind читає файли як звичайний текст і не розуміє конкатенацію, тож таких класів у тексті немає; потрібно писати класи повністю."),
    ("Які файли Tailwind не сканує автоматично?", "факт",
     ["detecting-classes-in-source-files"], r"node_modules", "Файли з .gitignore, node_modules, бінарні файли, CSS-файли і lock-файли."),
    ("Як примусово згенерувати клас, якого немає у вихідних файлах?", "як зробити",
     ["detecting-classes-in-source-files"], r"@source inline", "Директивою @source inline(\"назва-класу\")."),
    ("Як вимкнути Preflight?", "як зробити",
     ["preflight"], r"omit its import", "Імпортувати окремі частини Tailwind і пропустити імпорт tailwindcss/preflight.css."),
    ("Яка утиліта обрізає багаторядковий текст після заданої кількості рядків?", "клас або директива",
     ["line-clamp"], r"line-clamp-\x3cnumber>", "line-clamp-\u003cчисло>, наприклад line-clamp-3."),
    ("Який клас задає для елемента співвідношення сторін 16 до 9?", "клас або директива",
     ["aspect-ratio"], r"aspect-video", "aspect-video."),
    ("Яка утиліта додає анімацію обертання для індикатора завантаження?", "клас або директива",
     ["animation"], r"animate-spin", "animate-spin."),
    ("Чим автоматично відсортувати класи в HTML за рекомендованим порядком?", "як зробити",
     ["editor-setup"], r"Prettier plugin", "Офіційним плагіном Prettier для Tailwind CSS."),
    ("Скільки коштує ліцензія Tailwind Plus?", "без відповіді", [], None, None),
    ("Як підключити Tailwind CSS до проєкту на Angular?", "без відповіді", [], None, None),
    ("Хто створив Tailwind CSS?", "без відповіді", [], None, None),
    ("Яка версія React потрібна для роботи Tailwind CSS?", "без відповіді", [], None, None),
]

answerable = [(q, gold, evidence) for q, _, gold, evidence, _ in BENCHMARK if gold]
for question, _, gold, evidence, _ in BENCHMARK:
    # Перевіряємо, що вираз-доказ справді є в тексті одного з вказаних документів
    assert not gold or any(re.search(evidence, documents[doc]["text"]) for doc in gold), question

types = pd.Series([item[1] for item in BENCHMARK]).value_counts()
print(f"Питань: {len(BENCHMARK)}, з них без відповіді в документах: {len(BENCHMARK) - len(answerable)}")
print(types.to_string())

Питань: 29, з них без відповіді в документах: 4
як зробити            7
міграція              7
поняття               5
клас або директива    4
без відповіді         4
факт                  2


## 1.3. Індексація

**Модель ембедингів.** Qwen3-Embedding-0.6B: багатомовна, тому підходить для українських питань до англійських фрагментів. Вектори нормалізую, питання кодую з `prompt_name="query"`.

**Розбиття.** Перевіряю два варіанти:
- A: фрагменти по 1000 символів із перекриттям 200 (`RecursiveCharacterTextSplitter`), перед кожним фрагментом додаю назву сторінки; виходить 351 фрагмент;
- B: один фрагмент — один розділ сторінки за заголовками другого, третього і четвертого рівня. Розділи коротші за 200 символів приєдную до наступного, довші за 1500 ріжу на частини по 1200 з перекриттям 150. Перед фрагментом пишу «Сторінка › Розділ»; виходить 445 фрагментів.

**Сховище.** FAISS `IndexFlatIP`: за нормалізованих векторів скалярний добуток дорівнює косинусній схожості. Фрагментів менше тисячі, тому точного пошуку досить. Поруч будую BM25 для гібридного пошуку.

In [7]:
def fixed_chunks(size=1000, overlap=200):
    # Варіант A: розбиваємо весь текст сторінки за розміром, не зважаючи на заголовки
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap, separators=["\n\n", "\n", ". ", " ", ""], keep_separator="end"
    )
    chunks = []
    for name, doc in documents.items():
        for piece in splitter.split_text(doc["text"]):
            chunks.append({"doc_id": name, "title": doc["title"], "heading": "", "url": doc["url"],
                           "text": f"{doc['title']}\n{piece}"})
    return chunks


def section_chunks(max_size=1500):
    # Варіант B: один фрагмент — один розділ сторінки (за заголовками ##, ###, ####).
    # Дуже короткі розділи приєднуємо до наступного, надто довгі ріжемо за розміром
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200, chunk_overlap=150, separators=["\n\n", "\n", ". ", " ", ""], keep_separator="end"
    )
    chunks = []
    for name, doc in documents.items():
        buffer = ""
        for section in re.split(r"\n(?=#{2,4} )", doc["text"]):
            if len(section) < 200:
                buffer += section + "\n\n"
                continue
            section, buffer = buffer + section, ""
            heading = section.split("\n", 1)[0].lstrip("# ").strip() if section.startswith("#") else ""
            pieces = [section] if len(section) <= max_size else splitter.split_text(section)
            prefix = f"{doc['title']} › {heading}" if heading else doc["title"]
            for piece in pieces:
                chunks.append({"doc_id": name, "title": doc["title"], "heading": heading, "url": doc["url"],
                               "text": f"{prefix}\n{piece}"})
        if buffer.strip():
            chunks.append({"doc_id": name, "title": doc["title"], "heading": "", "url": doc["url"],
                           "text": f"{doc['title']}\n{buffer.strip()}"})
    return chunks


chunks_fixed = fixed_chunks()
chunks_sections = section_chunks()
for label, chunks in [("за розміром 1000/200", chunks_fixed), ("за розділами", chunks_sections)]:
    lengths = [len(chunk["text"]) for chunk in chunks]
    print(f"{label}: {len(chunks)} фрагментів, довжина: середня {np.mean(lengths):.0f}, медіана {np.median(lengths):.0f}, найбільша {max(lengths)} символів")

за розміром 1000/200: 351 фрагментів, довжина: середня 836, медіана 918, найбільша 1026 символів
за розділами: 445 фрагментів, довжина: середня 619, медіана 532, найбільша 1517 символів


In [8]:
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device, model_kwargs={"dtype": dtype})
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device, model_kwargs={"dtype": dtype})
CANDIDATES = 20


def tokenize(text):
    return re.findall(r"\w+", text.lower())


class Index:
    """Фрагменти + векторне сховище FAISS + BM25. Усі методи пошуку повертають номери фрагментів за спаданням релевантності."""

    def __init__(self, chunks):
        self.chunks = chunks
        vectors = embedder.encode([chunk["text"] for chunk in chunks], normalize_embeddings=True, batch_size=16)
        self.faiss = faiss.IndexFlatIP(vectors.shape[1])
        self.faiss.add(np.asarray(vectors, dtype="float32"))
        self.bm25 = BM25Okapi([tokenize(chunk["text"]) for chunk in chunks])

    def dense(self, query, k=10):
        vector = embedder.encode(query, normalize_embeddings=True, prompt_name="query")
        _, ids = self.faiss.search(np.asarray([vector], dtype="float32"), k)
        return [int(i) for i in ids[0]]

    def sparse(self, query, k=10):
        scores = self.bm25.get_scores(tokenize(query))
        return [int(i) for i in np.argsort(-scores)[:k]]

    def hybrid(self, query, k=10, c=60):
        scores = {}
        for ranking in (self.dense(query, CANDIDATES), self.sparse(query, CANDIDATES)):
            for rank, i in enumerate(ranking, start=1):
                scores[i] = scores.get(i, 0) + 1 / (c + rank)
        return sorted(scores, key=scores.get, reverse=True)[:k]

    def rerank(self, query, k=10):
        candidates = self.hybrid(query, CANDIDATES)
        scores = reranker.predict([(query, self.chunks[i]["text"]) for i in candidates])
        return [candidates[i] for i in np.argsort(-scores)[:k]]


start = time.time()
index_fixed = Index(chunks_fixed)
index_sections = Index(chunks_sections)
print(f"Індекси побудовано за {time.time() - start:.0f} с")

Індекси побудовано за 26 с


## 1.4. Ретрив

Порівнюю п'ять конфігурацій на 25 питаннях, що мають відповідь. Правильним вважаю фрагмент з потрібного документа, у тексті якого є вираз-доказ із бенчмарку.

- hit@k — частка питань, для яких правильний фрагмент є серед перших k результатів;
- MRR — середнє значення 1 / (місце першого правильного фрагмента).

Конфігурації: dense на фрагментах A (1000/200), dense на розділах B, BM25 на розділах, гібрид dense + BM25 з об'єднанням рангів RRF (c = 60) і гібрид з переранжуванням cross-encoder'ом bge-reranker-v2-m3 по 20 кандидатах.

In [9]:
def first_hit_rank(index, ids, gold, evidence):
    # Правильний фрагмент: з потрібного документа і з виразом-доказом
    for rank, i in enumerate(ids, start=1):
        chunk = index.chunks[i]
        if chunk["doc_id"] in gold and re.search(evidence, chunk["text"]):
            return rank
    return None


CONFIGS = {
    "dense, фрагменти 1000/200": (index_fixed, "dense"),
    "dense, за розділами": (index_sections, "dense"),
    "BM25, за розділами": (index_sections, "sparse"),
    "гібридний, за розділами": (index_sections, "hybrid"),
    "гібридний + reranker, за розділами": (index_sections, "rerank"),
}


def retrieval_ranks(index, method):
    return [first_hit_rank(index, getattr(index, method)(question, 10), gold, evidence)
            for question, gold, evidence in answerable]


def hit_rate(ranks, k):
    return np.mean([rank is not None and rank <= k for rank in ranks])


all_ranks, rows = {}, []
for name, (index, method) in CONFIGS.items():
    ranks = retrieval_ranks(index, method)
    all_ranks[name] = ranks
    rows.append({"конфігурація": name, "hit@1": hit_rate(ranks, 1), "hit@3": hit_rate(ranks, 3), "hit@5": hit_rate(ranks, 5),
                 "MRR": np.mean([1 / rank if rank else 0 for rank in ranks])})
metrics = pd.DataFrame(rows).round(2)
print(metrics.to_string(index=False))

                      конфігурація  hit@1  hit@3  hit@5  MRR
         dense, фрагменти 1000/200   0.60   0.84   0.92 0.73
               dense, за розділами   0.84   0.88   0.96 0.87
                BM25, за розділами   0.40   0.68   0.68 0.52
           гібридний, за розділами   0.68   0.84   0.84 0.77
гібридний + reranker, за розділами   0.88   1.00   1.00 0.93


In [10]:
# Де кожна конфігурація помилилась: питання, для яких правильного фрагмента немає на 1 місці
questions_answerable = [question for question, _, _ in answerable]
for name, ranks in all_ranks.items():
    bad = [(rank, question) for rank, question in zip(ranks, questions_answerable) if rank != 1]
    print(f"{name}: не на 1 місці — {len(bad)} з {len(ranks)}")
    for rank, question in bad:
        print(f"    місце {rank}: {question}")

dense, фрагменти 1000/200: не на 1 місці — 10 з 25
    місце 2: Чим у v4 замінили директиви @tailwind base, @tailwind components і @tailwind utilities?
    місце 5: Як виглядає префікс до класів у v4, якщо задати його tw?
    місце 3: Якої товщини кільце додавала утиліта ring у v3 і якою вона стала у v4?
    місце 9: Яка директива додає власні утиліти, що працюють з варіантами hover, focus і lg?
    місце 3: Для чого потрібна директива @reference у компонентах Vue чи Svelte?
    місце 2: Як увімкнути темну тему за допомогою класу dark замість налаштування системи?
    місце 2: Як додати власний брейкпоінт до теми?
    місце 4: Як примусово згенерувати клас, якого немає у вихідних файлах?
    місце None: Який клас задає для елемента співвідношення сторін 16 до 9?
    місце 2: Яка утиліта додає анімацію обертання для індикатора завантаження?
dense, за розділами: не на 1 місці — 4 з 25
    місце 3: Якою командою можна автоматично оновити проєкт із Tailwind v3 до v4?
    місце 4: Для чого 

**Висновок за ретривом.** Найкращою виявилась конфігурація з reranker'ом: hit@1 = 0,88, hit@3 = hit@5 = 1,00, MRR = 0,93. Її помилки — три питання, де правильний фрагмент стоїть на 2–3 місці (про `tailwind.config.js`, темну тему класом і container queries).

Розбиття за розділами краще за фрагменти по 1000 символів: hit@1 0,84 проти 0,60 і MRR 0,87 проти 0,73. У варіанті A розділ може розрізатися посередині, а у фрагмента немає заголовка, за яким його легко знайти.

BM25 найслабший (hit@1 = 0,40, у 8 питаннях з 25 правильного фрагмента немає навіть у десятці): питання українською, документація англійською, тож збігаються лише технічні терміни на кшталт `@theme`. Гібрид без reranker'а гірший за чистий dense на розділах (hit@5 0,84 проти 0,96): думаю, що слабкий BM25-ранжир тягне вниз правильні фрагменти. Reranker це виправляє, бо оцінює пару «питання — фрагмент» напряму. Для генерації беру цю конфігурацію і 5 фрагментів.

## 1.5. Генерація з цитуванням

Генеративна модель — Qwen3-4B-Instruct-2507, генерація жадібна (без семплінгу). Для відповіді беру 5 найкращих фрагментів із найкращої конфігурації ретриву. У промпті вимагаю відповідати лише за фрагментами, після кожного твердження ставити номер фрагмента в квадратних дужках, а якщо відповіді у фрагментах немає — відповісти фразою «У наданих документах немає відповіді.».

Оцінюю три речі:
- правильність відповіді: та сама модель порівнює її з еталонною відповіддю і каже «так» або «ні»;
- правильність цитування: серед цитованих номерів є фрагмент з документа, де лежить відповідь;
- відмови: на питання без відповіді система має відмовитися.

In [11]:
GENERATOR_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL_NAME, dtype=dtype).to(device)


def generate(messages, max_new_tokens=300):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(device)
    with torch.no_grad():
        output = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                              temperature=None, top_p=None, top_k=None)
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


total_tokens = sum(len(tokenizer(doc["text"])["input_ids"]) for doc in documents.values())
print(f"Усі {len(documents)} документів разом: {total_tokens} токенів")

Усі 39 документів разом: 81683 токенів


In [12]:
FINAL_INDEX, FINAL_METHOD, K = index_sections, "rerank", 5

SYSTEM_PROMPT = (
    "Ти — асистент, що відповідає на питання про Tailwind CSS за його документацією. "
    "Відповідай лише на основі наданих фрагментів. Фрагменти англійською, а відповідь пиши українською; "
    "назви класів, директив і код залишай так, як у документації. "
    "Після кожного твердження вказуй у квадратних дужках номер фрагмента, на якому воно ґрунтується, наприклад [2]. "
    "Якщо у фрагментах немає відповіді на питання, відповідай лише: «У наданих документах немає відповіді.» "
    "Відповідай коротко."
)


def rag(question, k=K, index=None, method=None, max_new_tokens=300):
    index, method = index or FINAL_INDEX, method or FINAL_METHOD
    chunks = [index.chunks[i] for i in getattr(index, method)(question, k)]
    context = "\n\n".join(f"[{n}] {chunk['text']}" for n, chunk in enumerate(chunks, start=1))
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Фрагменти документів:\n\n{context}\n\nПитання: {question}"},
    ]
    return generate(messages, max_new_tokens), chunks


for question in ["Які мінімальні версії браузерів потрібні для Tailwind CSS v4.0?", "Хто створив Tailwind CSS?"]:
    answer, chunks = rag(question)
    print(question)
    print(" ", answer)
    for n, chunk in enumerate(chunks, start=1):
        print(f"   [{n}] {chunk['doc_id']}: {chunk['text'][:80]!r}")

Які мінімальні версії браузерів потрібні для Tailwind CSS v4.0?
  Мінімальні версії браузерів для Tailwind CSS v4.0: Chrome 111, Safari 16.4, Firefox 128 [2].
   [1] upgrade-guide: 'Upgrade guide › Browser requirements\n### Browser requirements\n\nTailwind CSS v4.0'
   [2] compatibility: 'Compatibility › Browser support\n## Browser support\n\nTailwind CSS v4.0 is designe'
   [3] upgrade-guide: 'Upgrade guide\nTailwind CSS v4.0 is a new major version of the framework, so whil'
   [4] compatibility: 'Compatibility › Sass, Less, and Stylus\n## Sass, Less, and Stylus\n\nTailwind CSS v'
   [5] upgrade-guide: "Upgrade guide › Using the upgrade tool\n## Using the upgrade tool\n\nIf you'd like "
Хто створив Tailwind CSS?
  У наданих документах немає відповіді.
   [1] editor-setup: 'Editor setup › IntelliSense for VS Code\n## IntelliSense for VS Code\n\nThe officia'
   [2] compatibility: 'Compatibility › Sass, Less, and Stylus\n## Sass, Less, and Stylus\n\nTailwind CSS v'
   [3] compatibility: 

In [13]:
REFUSAL = "немає відповіді"
JUDGE_PROMPT = (
    "Порівняй відповідь системи з еталонною відповіддю на питання.\n\n"
    "Питання: {question}\nЕталонна відповідь: {reference}\nВідповідь системи: {answer}\n\n"
    "Чи містить відповідь системи ту саму інформацію, що й еталонна, без суперечностей? "
    "Відповідай одним словом: «так» або «ні»."
)


def judge(question, reference, answer):
    verdict = generate([{"role": "user", "content": JUDGE_PROMPT.format(question=question, reference=reference, answer=answer)}], max_new_tokens=5)
    return verdict.strip().lower().startswith("так")


def cited_docs(answer, chunks):
    numbers = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    return [chunks[n - 1]["doc_id"] for n in numbers if 0 < n <= len(chunks)]


def evaluate(question, kind, gold, evidence, reference):
    answer, chunks = rag(question)
    retrieved = [chunk["doc_id"] for chunk in chunks]
    if gold:
        correct = judge(question, reference, answer)
        citation_ok = any(doc in gold for doc in cited_docs(answer, chunks))
        found = first_hit_rank(FINAL_INDEX, getattr(FINAL_INDEX, FINAL_METHOD)(question, K), gold, evidence)
    else:
        correct, citation_ok, found = REFUSAL in answer.lower(), None, None
    return {"питання": question, "тип": kind, "правильно": correct, "цитування": citation_ok,
            "правильний фрагмент на місці": found, "документи": gold, "знайдено": retrieved, "відповідь": answer}


start = time.time()
evaluation = pd.DataFrame([evaluate(q, kind, gold, evidence, reference) for q, kind, gold, evidence, reference in BENCHMARK])
has_answer = evaluation["цитування"].notna()
print(f"Час оцінювання: {time.time() - start:.0f} с")
print(f"Правильних відповідей на питання з відповіддю: {evaluation[has_answer]['правильно'].mean():.0%} ({evaluation[has_answer]['правильно'].sum()} з {has_answer.sum()})")
print(f"Коректних цитувань: {evaluation[has_answer]['цитування'].astype(bool).mean():.0%} ({evaluation[has_answer]['цитування'].astype(bool).sum()} з {has_answer.sum()})")
print(f"Правильних відмов на питання без відповіді: {evaluation[~has_answer]['правильно'].mean():.0%} ({evaluation[~has_answer]['правильно'].sum()} з {(~has_answer).sum()})")
print()
print(evaluation[has_answer].groupby("тип")["правильно"].agg(["sum", "count"]).to_string())

Час оцінювання: 205 с
Правильних відповідей на питання з відповіддю: 96% (24 з 25)
Коректних цитувань: 92% (23 з 25)
Правильних відмов на питання без відповіді: 100% (4 з 4)

                    sum  count
тип                           
клас або директива    4      4
міграція              7      7
поняття               4      5
факт                  2      2
як зробити            7      7


**Результати.** Основна конфігурація: розділи, гібридний пошук із reranker'ом, 5 фрагментів у промпті. Оцінювання 29 питань тривало 205 с на T4.

| | правильних відповідей | коректних цитувань | правильних відмов |
|---|---|---|---|
| основна конфігурація | 24 з 25 (96 %) | 23 з 25 (92 %) | 4 з 4 |
| для порівняння: dense на фрагментах 1000/200 | 21 з 25 (84 %) | 21 з 25 (84 %) | 4 з 4 |

За типами питань основна конфігурація помилилась лише на одному питанні типу «поняття» (4 з 5), а «міграція», «як зробити», «клас або директива» і «факт» відповіла без помилок. На всі чотири питання без відповіді система відмовилась фразою «У наданих документах немає відповіді.», хоча ретрив для них знаходив схожі за темою фрагменти (наприклад, про налаштування редактора на питання про автора Tailwind).

Правильність відповіді оцінює та сама модель Qwen3-4B за еталонною відповіддю, тому проблемні відповіді я прочитав сам: у 1.6 розібрано, де оцінка слушна, а де ні.

## 1.6. Аналіз відмов

Спочатку виводжу всі проблемні відповіді: неправильні, без правильного цитування або хибні відмови чи хибні відповіді на питання без відповіді. Далі розбираю п'ять із них.

In [14]:
def describe(row):
    print(f"Питання ({row['тип']}): {row['питання']}")
    print(f"  правильний документ: {row['документи']}, знайдено: {row['знайдено']}, правильний фрагмент на місці: {row['правильний фрагмент на місці']}")
    print(f"  правильно: {row['правильно']}, цитування: {row['цитування']}")
    print(f"  відповідь: {row['відповідь']}\n")


failures = evaluation[(~evaluation["правильно"]) | (evaluation["цитування"] == False)]
print(f"Проблемних відповідей: {len(failures)}\n")
for _, row in failures.iterrows():
    describe(row)

Проблемних відповідей: 3

Питання (поняття): Для чого призначена директива @theme?
  правильний документ: ['functions-and-directives', 'theme'], знайдено: ['functions-and-directives', 'theme', 'functions-and-directives', 'theme', 'adding-custom-styles'], правильний фрагмент на місці: 1.0
  правильно: True, цитування: False
  відповідь: Директива `@theme` використовується для визначення особливих CSS-змінних, які впливають на створення утилітних класів у проекті. Вони дозволяють додавати нові кольори, розміри, перериви та інші дизайн-токени, які можуть бути використані у HTML як утилітні класи (наприклад, `bg-mint-500`). Крім того, Tailwind створює регулярні CSS-змінні для цих токенів, щоб можна було використовувати їх у вбудованих стилях. `@theme` вимагає бути визначено на рівні проекту, не вкладених у вибірки або медіа-запити, і використовується, коли потрібно, щоб дизайн-токени відповідали утилітним класам [4

Питання (поняття): Для чого потрібна директива @reference у компонентах Vu

In [15]:
# Той самий прогін, але з найслабшим ретривом (dense, фрагменти 1000/200): дивлюсь, як помилки ретриву доходять до відповіді
FINAL_INDEX, FINAL_METHOD = index_fixed, "dense"
weak = pd.DataFrame([evaluate(q, kind, gold, evidence, reference) for q, kind, gold, evidence, reference in BENCHMARK])
FINAL_INDEX, FINAL_METHOD = index_sections, "rerank"

print(f"Правильних відповідей: {weak[has_answer]['правильно'].sum()} з {has_answer.sum()}")
print(f"Коректних цитувань: {weak[has_answer]['цитування'].astype(bool).sum()} з {has_answer.sum()}")
print(f"Правильних відмов: {weak[~has_answer]['правильно'].sum()} з {(~has_answer).sum()}\n")

worse = weak[(~weak["правильно"]) & evaluation["правильно"]]
print(f"Відповіді, які з цим ретривом стали неправильними, хоча з основним були правильні: {len(worse)}\n")
for _, row in worse.iterrows():
    describe(row)

Правильних відповідей: 21 з 25
Коректних цитувань: 21 з 25
Правильних відмов: 4 з 4

Відповіді, які з цим ретривом стали неправильними, хоча з основним були правильні: 4

Питання (міграція): Як у v4 підключити старий файл tailwind.config.js?
  правильний документ: ['upgrade-guide'], знайдено: ['upgrade-guide', 'upgrade-guide', 'functions-and-directives', 'upgrade-guide', 'upgrade-guide'], правильний фрагмент на місці: 1.0
  правильно: False, цитування: False
  відповідь: Використовуйте директиву `@config` для підключення старого файлу `tailwind.config.js`:

```css
@config "../../tailwind.config.js";
```

У відповідності до [фрагмента 1] і [3].

Питання (поняття): Для чого потрібна директива @reference у компонентах Vue чи Svelte?
  правильний документ: ['functions-and-directives'], знайдено: ['functions-and-directives', 'upgrade-guide', 'functions-and-directives', 'compatibility', 'compatibility'], правильний фрагмент на місці: 3.0
  правильно: False, цитування: False
  відповідь: Дире

In [16]:
# Перевіряю виправлення для обрізаної відповіді про @theme: більший ліміт токенів
question = "Для чого призначена директива @theme?"
for limit in (300, 600):
    answer, _ = rag(question, max_new_tokens=limit)
    print(f"ліміт {limit}: довжина відповіді {len(tokenizer(answer)['input_ids'])} токенів, кінець: ...{answer[-90:]!r}")

ліміт 300: довжина відповіді 300 токенів, кінець: ...'пити, і використовується, коли потрібно, щоб дизайн-токени відповідали утилітним класам [4'
ліміт 600: довжина відповіді 399 токенів, кінець: ...'ласи, відмінюючи від `:root`, де змінні не пов’язані з утилітними класами [2]. [1] [4] [5]'


**Розбір відмов.** Випадки 1–3 — основна конфігурація, 4–6 — прогін із слабшим ретривом (dense на фрагментах 1000/200): там чотири відповіді, правильні в основній конфігурації, стали хибними. Три з них розібрано нижче, четверта (про `@reference`) повторює випадок 2 і окремо не описується.

**1. Відповідь обрізана (питання про `@theme`, основна конфігурація).** Тип: генерація, ліміт токенів. Відповідь упирається в 300 токенів і обривається на «[4», тому цитування не зараховано, хоча сама відповідь правильна. Промпт просив відповідати коротко, але модель не вклалась. Виправлення: збільшити ліміт. Перевірка вище: з лімітом 600 відповідь займає 399 токенів і закінчується повністю.

**2. Цитата на чужий фрагмент (питання про `@reference`, основна конфігурація).** Тип: хибне цитування. Правильний фрагмент стояв першим, а модель послалась на [2] — фрагмент зі сторінки про оновлення. Відповідь за змістом правильна, але читач, що відкриє джерело [2], там її не знайде. Виправлення: після генерації перевіряти, що цитований фрагмент справді містить ключові терміни відповіді, або вимагати від моделі коротку дослівну цитату разом із номером.

**3. Незграбний переклад терміна (питання про mobile-first, основна конфігурація).** Тип: якість мови, а також суворість судді. У відповіді утиліти без префікса названо «непередбаченими (не присутніми)» (англійське unprefixed), решта відповіді близька до еталонної. Суддя поставив «ні», і за змістом це спірно. Виправлення: додати в промпт короткий глосарій («без префікса», «брейкпоінт») і перевіряти відповіді сильнішим суддею.

**4. Правильної сторінки немає серед знайдених (питання про співвідношення сторін 16:9, dense на фрагментах 1000/200).** Тип: промах ретриву. Усі п'ять фрагментів зі сторінки про hover і focus, сторінки aspect-ratio серед них немає, тому модель відмовилась. Відмова чесна, але питання мало відповідь. З розділами і reranker'ом правильний фрагмент був на 1 місці. Виправлення: ретрив за розділами з переранжуванням.

**5. Неповна відповідь (темна тема за класом `dark`, dense на фрагментах 1000/200).** Тип: модель взяла з правильного фрагмента не те. Фрагмент стояв другим, але відповідь складається з JavaScript-прикладу з `localStorage`, а перевизначення варіанта `@custom-variant dark`, яке і є відповіддю, не згадано. Виправлення: у промпті вимагати спочатку назвати головний крок, а в розбитті відокремлювати приклади коду від пояснення.

**6. Формат цитування і неповна відповідь (питання про `tailwind.config.js`, dense на фрагментах 1000/200).** Тип: порушення формату. Модель написала «[фрагмента 1] і [3]» замість «[1]», і моя перевірка цитат їх не розпізнала. До того ж у відповіді немає, що файл конфігурації v4 сам не підхоплює, тому суддя міг поставити «ні» небезпідставно. Виправлення: у промпті навести приклад формату, а при розборі приймати і такі записи.

**Що змінив би в першу чергу.** Ліміт токенів (випадок 1) і перевірку цитат (2 і 6) — це найдешевші виправлення. Окрема проблема — сам суддя: оцінювати відповіді іншою, сильнішою моделлю або вручну на вибірці.

## 1.7. Демонстрація

Простий застосунок на Gradio: вводжу питання, отримую відповідь українською і список джерел із назвою сторінки, розділом, посиланням на tailwindcss.com і початком фрагмента. Повзунок задає, скільки фрагментів передати моделі. Застосунок запускається просто в ноутбуці з публічним посиланням (`share=True`), яке працює, поки відкрита сесія Colab.

In [17]:
!pip install -q gradio

In [18]:
import gradio as gr


def ask(question, k):
    if not question.strip():
        return "Введіть питання.", ""
    answer, chunks = rag(question.strip(), int(k))
    sources = "\n\n".join(
        f"**[{n}] [{chunk['title']}]({chunk['url']})**" + (f" — розділ «{chunk['heading']}»" if chunk["heading"] else "")
        + "\n\n> " + chunk["text"].split("\n", 1)[1][:350].replace("\n", "\n> ") + "…"
        for n, chunk in enumerate(chunks, start=1)
    )
    return answer, sources


with gr.Blocks(title="Питання до документації Tailwind CSS") as demo:
    gr.Markdown("# Питання до документації Tailwind CSS v4\nВідповідь складається з фрагментів документації, номери в квадратних дужках вказують на джерела нижче.")
    question = gr.Textbox(label="Питання", placeholder="Наприклад: Як вимкнути Preflight?", lines=2)
    k = gr.Slider(1, 8, value=K, step=1, label="Скільки фрагментів передати моделі")
    button = gr.Button("Запитати", variant="primary")
    answer_box = gr.Textbox(label="Відповідь", lines=6)
    sources_box = gr.Markdown()
    gr.Examples(["Як вимкнути Preflight?", "Яка команда оновлює проєкт із v3 до v4?", "Як увімкнути темну тему класом?", "Скільки коштує Tailwind Plus?"], inputs=question)
    button.click(ask, [question, k], [answer_box, sources_box])
    question.submit(ask, [question, k], [answer_box, sources_box])

demo.launch(share=True, prevent_thread_lock=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cdf4c84802be5cbdf1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 1.8. Мій звіт

**Корпус і чому потрібен RAG:** документація Tailwind CSS v4 (репозиторій tailwindlabs/tailwindcss.com, папка `src/docs`, коміт `7f92c22`): 39 сторінок у MDX, після очищення 251 КБ тексту. Ліцензія репозиторію не відкрита, тому корпус не лежить у моєму репозиторії, а ноутбук завантажує його за комітом. Питає розробник, який верстає на Tailwind, українською, а документація англійська. У промпт усе не покласти, fine-tuning довелося б повторювати після кожного релізу і він не вказує джерело, а RAG вказує сторінку і оновлюється заміною корпусу.

**Бенчмарк:** 29 питань українською, складених до налаштування системи: 25 з відповіддю в документах і 4 без відповіді. За типами: як зробити 7, міграція 7, поняття 5, клас або директива 4, факт 2, без відповіді 4. До кожного питання записано документ із відповіддю, вираз-доказ і еталонну відповідь.

**Індексація (модель ембедингів, розбиття, сховище):** Qwen3-Embedding-0.6B з нормалізованими векторами; два розбиття: фрагменти 1000 символів із перекриттям 200 (351 фрагмент) і розбиття за розділами сторінки (445 фрагментів, у середньому 619 символів); сховище FAISS `IndexFlatIP`, поруч BM25 для гібридного пошуку.

**Результати ретриву для різних конфігурацій:**

| конфігурація | hit@1 | hit@3 | hit@5 | MRR |
|---|---|---|---|---|
| dense, фрагменти 1000/200 | 0,60 | 0,84 | 0,92 | 0,73 |
| dense, за розділами | 0,84 | 0,88 | 0,96 | 0,87 |
| BM25, за розділами | 0,40 | 0,68 | 0,68 | 0,52 |
| гібридний (RRF), за розділами | 0,68 | 0,84 | 0,84 | 0,77 |
| гібридний + reranker, за розділами | 0,88 | 1,00 | 1,00 | 0,93 |

**Генеративна модель і результати відповідей:** Qwen3-4B-Instruct-2507, жадібна генерація, 5 фрагментів з найкращої конфігурації. Правильних відповідей 24 з 25 (96 %), коректних цитувань 23 з 25 (92 %), правильних відмов на питання без відповіді 4 з 4. З ретривом dense на фрагментах 1000/200 було 21 з 25 правильних відповідей і 21 з 25 цитувань.

**Аналіз відмов:** розібрано шість випадків: обрізана відповідь через ліміт токенів, цитата на чужий фрагмент, незграбний переклад терміна (і суворий суддя), промах ретриву, неповна відповідь із правильного фрагмента, порушення формату цитування. Для кожного в 1.6 є тип і виправлення; одне з них (ліміт 600 токенів замість 300) я перевірив: відповідь про `@theme` стала повною.

**Висновок:** на цьому корпусі найбільше дало розбиття за розділами (hit@1 з 0,60 до 0,84) і reranker (до 0,88, hit@5 = 1,00); BM25 для українських питань до англійських документів слабкий, а гібрид без reranker'а гірший за чистий dense. Генерація майже не помиляється, коли правильний фрагмент у п'ятірці, а решта проблем — обрізання відповіді та хибні цитати. Обмеження: бенчмарк малий (25 питань з відповіддю), правильність оцінює та сама модель, що генерує. Застосунок на Gradio запускається в розділі 1.7.